# Fabric Workspace Recycle Bin Cleaner

**Utilitario genérico para inventariar y eliminar permanentemente elementos recuperables de Microsoft Fabric.**

1. Introduce el identificador de tu Workspace.
2. Ejecuta el inventario y revisa los candidatos.
3. Opcionalmente ajusta filtros.
4. La eliminación está deshabilitada por defecto (`DRY_RUN=True`). Para ejecutarla, cambia a `False` y escribe el texto de confirmación indicado.

⚠️ **La eliminación permanente es irreversible desde Fabric.** Requiere permisos y scopes adecuados (habitualmente rol Workspace Admin). El notebook no elimina elementos activos.

Documentación: [List Recoverable Items](https://learn.microsoft.com/en-us/rest/api/fabric/core/recoverable-items/list-recoverable-items) · [Delete Recoverable Item](https://learn.microsoft.com/en-us/rest/api/fabric/core/recoverable-items/delete-recoverable-item) · [NotebookUtils](https://learn.microsoft.com/en-us/fabric/data-engineering/notebookutils/notebookutils-credentials)


In [ ]:
# CELDA 1 — CONFIGURACIÓN
WORKSPACE_ID = "REEMPLAZAR_CON_WORKSPACE_ID"   # GUID del workspace de destino

# Filtros vacíos = todos los elementos recuperables
TYPES = []              # Ej.: ["Warehouse", "Lakehouse"]
NAME_CONTAINS = ""      # Ej.: "Contoso" (sin distinguir mayúsculas)

DRY_RUN = True           # Modo seguro; True = NO elimina nada
CONFIRMATION = ""       # Para borrar: "DELETE <WORKSPACE_ID> <N>" generado más abajo

# No intentar por separado los hijos si sus padres se seleccionan
SKIP_CHILDREN_OF_SELECTED_PARENTS = True
REQUEST_TIMEOUT = 30
MAX_RETRIES = 4


In [ ]:
# CELDA 2 — CONEXIÓN Y FUNCIONES
import time
import uuid
from urllib.parse import urlparse
import requests
import pandas as pd

BASE_URL = "https://api.fabric.microsoft.com/v1"

def validate_workspace_id(workspace_id):
    try:
        return str(uuid.UUID(workspace_id))
    except (ValueError, TypeError) as exc:
        raise ValueError("Introduce un WORKSPACE_ID válido (GUID) en la celda 1.") from exc

def api_call(method, url, **kwargs):
    # El token se solicita por operación para tolerar ejecuciones largas.
    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, headers=headers, timeout=REQUEST_TIMEOUT, **kwargs)
        except requests.RequestException:
            if attempt >= MAX_RETRIES:
                raise
            time.sleep(min(2 ** attempt, 30))
            continue
        if response.status_code in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
            retry_after = response.headers.get("Retry-After")
            try:
                delay = max(1, min(float(retry_after), 60))
            except (ValueError, TypeError):
                delay = min(2 ** attempt, 30)
            time.sleep(delay)
            continue
        if not response.ok:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")
        return response
    raise RuntimeError("Sin respuesta después de los reintentos")

def list_recoverable_items(workspace_id):
    url = f"{BASE_URL}/workspaces/{workspace_id}/recoverableItems"
    items = []
    visited = set()
    while url:
        if url in visited:
            raise RuntimeError("Paginación repetida detectada")
        visited.add(url)
        payload = api_call("GET", url).json()
        items.extend(payload.get("value", []))
        next_url = payload.get("continuationUri")
        if not next_url and payload.get("continuationToken"):
            from urllib.parse import urlencode
            next_url = f"{BASE_URL}/workspaces/{workspace_id}/recoverableItems?" + urlencode({"continuationToken": payload["continuationToken"]})
        if next_url:
            parsed = urlparse(next_url)
            if parsed.scheme != "https" or parsed.netloc != "api.fabric.microsoft.com" or not parsed.path.startswith(f"/v1/workspaces/{workspace_id}/recoverableItems"):
                raise RuntimeError("La API devolvió una URL de paginación inesperada")
        url = next_url
    return list({i["id"]: i for i in items}.values())

workspace_id = validate_workspace_id(WORKSPACE_ID)
items = list_recoverable_items(workspace_id)
print(f"Elementos recuperables encontrados: {len(items)}")


In [ ]:
# CELDA 3 — INVENTARIO, FILTROS Y PLAN DE ELIMINACIÓN
records = []
for item in items:
    if TYPES and item.get("type") not in TYPES:
        continue
    if NAME_CONTAINS and NAME_CONTAINS.casefold() not in item.get("displayName", "").casefold():
        continue
    records.append(item)

selected_ids = {i["id"] for i in records}
# Si un padre está seleccionado, su hijo no se solicita por separado.
# No se asume que esta regla funcione para todas las relaciones de Fabric.
if SKIP_CHILDREN_OF_SELECTED_PARENTS:
    candidates = [i for i in records if i.get("parentItemId") not in selected_ids]
else:
    candidates = records[:]

candidates = sorted(candidates, key=lambda i: (i.get("type", ""), i.get("displayName", ""), i["id"]))
preview = pd.DataFrame([{
    "Nombre": i.get("displayName", ""),
    "Tipo": i.get("type", ""),
    "Item ID": i["id"],
    "Parent ID": i.get("parentItemId", ""),
    "Retención hasta": i.get("retentionExpirationDateTime", "")
} for i in candidates], columns=["Nombre", "Tipo", "Item ID", "Parent ID", "Retención hasta"])

display(preview)
print(f"Total en papelera: {len(items)} | Coinciden con filtros: {len(records)} | Solicitudes DELETE previstas: {len(candidates)}")
expected_confirmation = f"DELETE {workspace_id} {len(candidates)}"
print(f"DRY_RUN = {DRY_RUN}")
print("Confirmación requerida para borrar:", expected_confirmation)


In [ ]:
# CELDA 4 — EJECUCIÓN CONTROLADA
# Ejecuta esta celda solo después de revisar el inventario anterior.
# Para eliminar: establece DRY_RUN=False y CONFIRMATION exactamente como se imprimió.

if DRY_RUN:
    print("SIMULACIÓN: no se ejecutó ninguna eliminación.")
elif not candidates:
    print("No hay elementos candidatos; nada que eliminar.")
elif CONFIRMATION != expected_confirmation:
    raise ValueError("Confirmación incorrecta. No se eliminó ningún elemento.")
else:
    # Revalidar el conjunto antes de ejecutar, para impedir borrar una selección que cambió.
    fresh = list_recoverable_items(workspace_id)
    fresh_ids = {i["id"] for i in fresh}
    if any(i["id"] not in fresh_ids for i in candidates):
        raise RuntimeError("La papelera cambió. Ejecuta nuevamente la celda 3 para revisar el inventario.")
    results = []
    for item in candidates:
        item_id = item["id"]
        url = f"{BASE_URL}/workspaces/{workspace_id}/recoverableItems/{item_id}"
        try:
            response = api_call("DELETE", url)
            status = "Eliminado"
            details = f"HTTP {response.status_code}"
        except Exception as exc:
            status = "Error"
            details = str(exc)[:500]
        results.append({"Nombre": item.get("displayName"), "Tipo": item.get("type"), "ID": item_id, "Estado": status, "Detalle": details})
        print(f"{status}: {item.get('displayName')} ({item_id})")
    result_df = pd.DataFrame(results)
    display(result_df)
    print("Solicitudes exitosas:", (result_df["Estado"] == "Eliminado").sum())
    print("Errores:", (result_df["Estado"] == "Error").sum())
    print("Vuelve a ejecutar la celda 2 para consultar el estado final de la papelera.")


## Consideraciones de seguridad
- El notebook actúa sobre el **Workspace ID especificado**, independientemente del Workspace en el que esté guardado el notebook.
- En modo `DRY_RUN`, no ejecuta solicitudes DELETE.
- El borrado permanente es irreversible. Confirma el nombre y el ID del Workspace antes de habilitarlo.
- Las solicitudes pueden fallar por permisos insuficientes, dependencias, estados transitorios o límites de la API. Revisa el informe y vuelve a consultar la papelera.
- Si el notebook se ejecuta bajo service principal, el token `getToken("pbi")` puede tener scopes limitados; consulta los requisitos de autenticación y usa MSAL si corresponde.
- Los hijos pueden aparecer como elementos recuperables asociados a un padre. El filtro de hijos evita solicitudes redundantes, pero **no asegura** el borrado de todos los tipos de relaciones.
- No proporciona auditoría persistente: exporta el resultado a un almacenamiento aprobado si lo necesitas como control operativo.
